In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, when

In [2]:
# Create Spark session
spark = SparkSession.builder \
                    .appName("Healthcare Claims Ingestion") \
                    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/12 08:40:51 INFO SparkEnv: Registering MapOutputTracker
26/05/12 08:40:51 INFO SparkEnv: Registering BlockManagerMaster
26/05/12 08:40:51 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/05/12 08:40:51 INFO SparkEnv: Registering OutputCommitCoordinator


In [3]:
# configure variables
BUCKET_NAME = "healthcare-bucket-minhld"
CLAIMS_BUCKET_PATH = f"gs://{BUCKET_NAME}/landing/claims/*.csv"

In [4]:
# BigQuery Configuration
BQ_PROJECT = "healthcare-496102"
BQ_TABLE = f"{BQ_PROJECT}.bronze_dataset.claims"
TEMP_GCS_BUCKET = f"{BUCKET_NAME}/temp/"

In [5]:
# read from claims source
claims_df = spark.read.csv(CLAIMS_BUCKET_PATH, header=True)

In [6]:
# adding hospital source for future reference
claims_df = (claims_df
                .withColumn("datasource", 
                              when(input_file_name().contains("hospital2"), "hosb")
                             .when(input_file_name().contains("hospital1"), "hosa").otherwise("None")))

In [7]:
# dropping dupplicates if any
claims_df = claims_df.dropDuplicates()

In [8]:
# write to bigquery
(claims_df.write
            .format("bigquery")
            .option("table", BQ_TABLE)
            .option("temporaryGcsBucket", TEMP_GCS_BUCKET)
            .mode("overwrite")
            .save())

26/05/12 08:43:46 ERROR YarnClientSchedulerBackend: YARN application has exited unexpectedly with state KILLED! Check the YARN application logs for more details.
26/05/12 08:43:46 ERROR YarnClientSchedulerBackend: Diagnostics message: Application application_1778569936395_0008 was killed by user ducmi at 10.148.0.5
26/05/12 08:43:46 ERROR ApplicationMaster: Exception from Reporter thread.
org.apache.hadoop.yarn.exceptions.ApplicationAttemptNotFoundException: Application attempt appattempt_1778569936395_0008_000001 doesn't exist in ApplicationMasterService cache.
	at org.apache.hadoop.yarn.server.resourcemanager.ApplicationMasterService.allocate(ApplicationMasterService.java:408)
	at org.apache.hadoop.yarn.api.impl.pb.service.ApplicationMasterProtocolPBServiceImpl.allocate(ApplicationMasterProtocolPBServiceImpl.java:60)
	at org.apache.hadoop.yarn.proto.ApplicationMasterProtocol$ApplicationMasterProtocolService$2.callBlockingMethod(ApplicationMasterProtocol.java:106)
	at org.apache.hadoo